 # <center>  AI ASSISTANT ON WHATSAPP 

# `*Initialize the packages*`

## Since I used gemini as Engine I found this key API at website Google Studio AI

In [1]:
from google.genai import client
from google.genai import types
# Initialize the Gemini client with your API key
ai = client.Client(api_key="GEMINI_API_KEY")
# Send a simple test prompt to the fast gemini-2.5-flash model
response = ai.models.generate_content(
    model='gemini-2.5-flash',
    contents='Respond with: System test successful!',
)
print(response.text)

System test successful!


# `*Test if My AI works Well !*`

In [2]:
import time
from google.genai import client

# Initialize client session with your key
ai = client.Client(api_key="GEMINI_API_KEY")
chat = ai.chats.create(model="gemini-2.5-flash")

try:
    # First interaction: Tell it your major explicitly
    print("Sending message 1...")
    response1 = chat.send_message("Hello! I am a machine learning engineering student, who are you?")
    print("gloire AI 1 :", response1.text)
    
    # ⏳ Add a 5-second pause to let the API free-tier quota cool down
    print("\n[System] Waiting 5 seconds to avoid rate limits...")
    time.sleep(5)
    
    # Second interaction: Now test its memory
    print("\nSending message 2...")
    response2 = chat.send_message("What major did I say I am studying?")
    print("gloire AI 2:", response2.text)

except Exception as e:
    print(f"\n[Error caught]: {e}")

Sending message 1...
gloire AI 1 : Hello there!

I am a large language model, trained by Google.

My purpose is to be helpful and informative across a wide range of topics. I can answer questions, generate text, provide summaries, assist with coding, brainstorm ideas, and much more.

It's great to connect with an ML engineering student – you're in a fascinating field! How can I help you today?

[System] Waiting 5 seconds to avoid rate limits...

Sending message 2...
gloire AI 2: You said you are studying **Machine Learning Engineering**.


# `My Final AI with all APIs`

### `Flask` provides the link : `http://127.0.0.1:5000` that is mean my python code is on port 5000 of my PC and I used `ngrok` to provide to my AI the internet Link(url) with this command on my DOS `ngrok http 5000` and generate the url for my Python Code (`https://dominion-stimuli-demystify.ngrok-free.dev`)  that I will set up on Twilio API to allow a WhatsApp user to communicate with my AI on my PC 

In [ ]:
import os
from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse
from google.genai import client
from google.genai import types  # Imported to handle system instruction rules

app = Flask(__name__)

# engine
API_KEY = "GEMINI_API_KEY"
ai = client.Client(api_key=API_KEY)

user_sessions = {}

# my customized developer bio rules
BOT_INSTRUCTIONS = (
    "You are a helpful AI assistant. At the end of your very first message to a user, "
    "or when asked who created you, explicitly state: 'I am an LLM developed by Gloire Ahadi, "
    "a Mechanical Engineer.' CRITICAL RULE: You must keep your entire response concise. "
    "Never write more than 10 sentences total under any circumstance."
)

@app.route("/whatsapp", methods=['POST'])
def whatsapp_bot():
    incoming_msg = request.values.get('Body', '').strip()
    sender_phone = request.values.get('From', '') 
    
    print(f"\n[Incoming Route] Received from {sender_phone}: '{incoming_msg}'")
    
    # Create the session with system instructions applied
    if sender_phone not in user_sessions:
        print(f"[Session Control] Initializing fresh Gemini context memory for {sender_phone}")
        user_sessions[sender_phone] = ai.chats.create(
            model="gemini-2.5-flash",
            config=types.GenerateContentConfig(
                system_instruction=BOT_INSTRUCTIONS
            )
        )
    
    active_chat = user_sessions[sender_phone]
    
    try:
        gemini_response = active_chat.send_message(incoming_msg)
        reply_text = gemini_response.text
    except Exception as e:
        print(f"[API Critical Error] Failed to fetch response from Gemini: {e}")
        reply_text = "I encountered a minor network error processing that request. Please try again in a few seconds."
    
    twilio_resp = MessagingResponse()
    twilio_resp.message(reply_text)
    
    return str(twilio_resp)

if __name__ == "__main__":
    app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
